# AiJockey — Colab Pipeline (no-restart edition)

All heavy work runs as `!python ...` subprocess → fresh interpreter every time → no kernel restart needed.

**Time**: ~10 min for 5-min mix from 5 clips on T4.

## 1. GPU + system binaries

In [ ]:
!nvidia-smi | head -20
!apt-get install -y rubberband-cli ffmpeg > /dev/null 2>&1
!rubberband --version 2>&1 | head -1
!ffmpeg -version 2>&1 | head -1

## 2. Get code

Pick ONE option. Comment the others.

In [ ]:
# === OPTION A: git clone (works once repo is public) ===
!git clone https://github.com/architagrawal/aiJockey.git
%cd aiJockey

# === OPTION A2: private repo via Colab Secrets + PAT ===
# 1. Create PAT at https://github.com/settings/tokens (scope: repo)
# 2. Colab sidebar -> key icon -> + New secret -> name: GITHUB_TOKEN, value: <paste>
# from google.colab import userdata
# token = userdata.get('GITHUB_TOKEN')
# !git clone https://{token}@github.com/architagrawal/aiJockey.git
# %cd aiJockey

# === OPTION B: Drive sync ===
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/AiJockey /content/aiJockey
# %cd /content/aiJockey

# === OPTION C: ZIP upload ===
# from google.colab import files
# uploaded = files.upload()  # select AiJockey.zip
# !unzip -o AiJockey.zip -d /content/
# %cd /content/AiJockey

import os
print('cwd:', os.getcwd())
!ls

## 3. Python deps

No restart needed — pipeline runs via `!python` subprocess which picks up fresh interpreter state.

In [ ]:
# numpy<2 + cython<3 first (madmom needs both at build time)
!pip install --force-reinstall --no-deps "numpy<2.0"
!pip install --no-build-isolation --upgrade "cython<3"

# Audio + ML libs
!pip install demucs==4.0.1 librosa==0.10.2 soundfile==0.12.1 \
    pyrubberband pyloudnorm==0.1.1 scipy tqdm transformers

# madmom — Py 3.12 needs git source build
!pip install --no-build-isolation git+https://github.com/CPJKU/madmom.git

# CLAP runs via transformers.ClapModel (see src/clap_wrapper.py)
# laion-clap intentionally skipped (build fails on Py 3.12)

print('\n=== INSTALL DONE. No restart needed — subprocess will pick up fresh state. ===')

## 4. Verify install via subprocess

Runs in a NEW Python process — sees fresh numpy + new packages without kernel restart.

In [ ]:
!python -c "\
import importlib, torch; \
libs=['torch','torchaudio','demucs','madmom','librosa','soundfile','pyrubberband','pyloudnorm','transformers','scipy','numpy']; \
[print(f'  OK  {n:14s} ' + getattr(importlib.import_module(n),'__version__','?')) for n in libs]; \
print(f'cuda: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')"

## 5. Drop your clips

Upload 5 audio files (.wav/.mp3, ~3 min each) into `clips/` via Colab Files panel,
or use the uploader below.

In [ ]:
import os
os.makedirs('clips', exist_ok=True)

# === OPTION 1: ZIP upload (recommended — bulk) ===
# On Windows: Compress-Archive -Path clips_demo -DestinationPath clips_demo.zip -Force
# Then run this:
from google.colab import files
print('Pick clips_demo.zip from your computer:')
uploaded = files.upload()
for fn in uploaded:
    if fn.endswith('.zip'):
        get_ipython().system(f'unzip -oq "{fn}" -d /content/extracted/')
        # Move all .wav/.mp3 files into clips/ (handle nested dirs)
        get_ipython().system('find /content/extracted -type f \\( -iname "*.wav" -o -iname "*.mp3" -o -iname "*.flac" -o -iname "*.m4a" \\) -exec cp {} clips/ \\;')

# === OPTION 2: individual file upload (skip if ZIP done) ===
# uploaded = files.upload()
# import shutil
# for fn in uploaded:
#     shutil.move(fn, f'clips/{fn}')

clips = [f for f in os.listdir('clips')
         if f.lower().endswith(('.wav','.mp3','.flac','.m4a','.ogg'))]
print(f'\n{len(clips)} clips ready:')
for c in clips[:30]:
    print('  ', c)
if len(clips) > 30:
    print(f'  ... ({len(clips)-30} more)')

## 6. Pipeline — subprocess (fresh interpreter)

In [ ]:
# 5-min test mix. Bump --duration to 1800 for full 30-min set after first pass.
# To use trained models (run train_classifier.ipynb / train_clap_compat.ipynb first):
#   add: --classifier checkpoints/technique_classifier.pt
#   add: --compat_head checkpoints/clap_compat_head.pt
!python src/main.py all \
    --clips clips/ \
    --cache cache/ \
    --out_dir output/ \
    --device cuda \
    --duration 300 \
    --max_clips 200 \
    --surprises 10 \
    --callbacks 1 \
    --lufs -9

In [ ]:
# Optional: zero-shot genre classify (CLAP text-audio similarity).
# Tags each cached clip with detected genre. Useful for visualization
# and as a feature for downstream training.
!python src/training/genre_classify.py --cache cache/

## 7. Audition

IPython display — no numpy in-kernel needed.

In [ ]:
from IPython.display import Audio, display
import os
for path in ('output/raw_mix.wav', 'output/final_mix.wav'):
    if os.path.exists(path):
        print(path)
        display(Audio(path))

## 8. Inspect timeline

JSON only — no numpy.

In [ ]:
import json
with open('output/timeline.json') as f:
    tl = json.load(f)['timeline']
print(f'{len(tl)} entries')
for i, e in enumerate(tl):
    seg = e['segment']
    tech = e['transition_in']
    print(f"  {i:2d}. {e['clip_id']:20s} "
          f"[{seg.get('type','?'):10s}] "
          f"key={e['target_key']:>3s} bpm={e['target_bpm']:.1f} "
          f"-> {tech.get('name','?')} (bars={tech.get('bars','-')})")

## 9. Eval metrics — subprocess

In [ ]:
!python src/main.py eval \
    --mix output/final_mix.wav \
    --timeline output/timeline.json \
    --reference_cache cache/

## 10. Save to Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, os, datetime as dt
# stamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
# dst = f'/content/drive/MyDrive/AiJockey_runs/{stamp}'
# os.makedirs(dst, exist_ok=True)
# shutil.copytree('output', dst, dirs_exist_ok=True)
# print('saved to', dst)

## 11. Iterate — re-plan without re-analyzing

Analysis cached. Tweak params, rerun plan + execute + master. All subprocess.

In [ ]:
# Iterate without re-analyzing. Tweak --duration / --surprises / --callbacks.
!python src/main.py plan --cache cache/ --duration 1800 --surprises 10 --callbacks 2 --max_clips 200
!python src/main.py execute --timeline output/timeline.json --cache cache/ --out output/raw_mix_v2.wav
!python src/main.py master --in_path output/raw_mix_v2.wav --out output/final_mix_v2.wav --lufs -9

from IPython.display import Audio
Audio('output/final_mix_v2.wav')

## Troubleshooting

| Symptom | Fix |
|---------|-----|
| `madmom` install fails | re-run install cell — it pins cython<3 + numpy<2 first |
| `OOM` on Demucs | edit `src/analyze.py` Analyzer init: `demucs_model='mdx_extra_q'` |
| Mix is silent | inspect `cache/<clip>.json` — if `tempo=0`, beat detection failed for that clip |
| `audios` keyword error | already patched in `src/clap_wrapper.py`, ensure latest code uploaded |
| `BaseModelOutputWithPooling no shape` | already patched in `src/clap_wrapper.py`, ensure latest code uploaded |
| Colab disconnects | mount Drive (cell 10), persist cache/ + output/ |

## After first listen

1. Identify weakest transition in timeline (cell 8)
2. Tune `transition_score` weights in `src/planner.py`
3. Re-run plan + execute (cell 11) — analyze cached, fast